In [ ]:
import eval_common as _ec
_ec.set_axis("mechanism")
AXIS = _ec.AXIS
print("AXIS =", AXIS, "| categories:", _ec.MECH_ORDER)

AXIS = mechanism | categories: ['Reliability', 'Bias & Fairness', 'Privacy, Confidentiality & Infringement', 'Security & Misuse', 'Autonomous Actions', 'Governance, Oversight & Explainability']


In [ ]:
import eval_common as _ec
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
from sklearn.linear_model import MultiTaskElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import eval_common as _ec

RANDOM_SEED = 42

DATA_PATH = Path("incidents_classified.csv")
OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

DATE_COL = "date"
TARGET_COL = "mechanism"

MECH_KEEP = _ec.NAMED_MECHS
MECH_ORDER = _ec.MECH_ORDER
df = _ec.load_incidents()
df["mech7"] = df["mechanism"]  
print(df["mech7"].value_counts().reindex(MECH_ORDER))

REFERENCE_CATEGORY = _ec.REFERENCE_CATEGORY

LAGS = [1, 2, 3, 6, 12]
ROLLING_WINDOWS = [3, 6, 12]

ALPHA_GRID = [0.001, 0.01, 0.1, 1.0]
L1_RATIO_GRID = [0.1, 0.5, 0.9]

PSEUDOCOUNT = 0.5
INNER_VALIDATION_MONTHS = 12

df = _ec.load_incidents() 

df["mech7"] = df[TARGET_COL].where(df[TARGET_COL].isin(MECH_ORDER), MECH_ORDER[-1])


print("Rows:", len(df))
print("Date range:", df[DATE_COL].min(), "to", df[DATE_COL].max())
print(df["mech7"].value_counts().reindex(MECH_ORDER))

mech7
Reliability                                218
Bias & Fairness                            168
Privacy, Confidentiality & Infringement    223
Security & Misuse                          664
Autonomous Actions                         222
Governance, Oversight & Explainability      82
Name: count, dtype: int64
Rows: 1577
Date range: 2016-01-01 00:00:00 to 2026-07-30 00:00:00
mech7
Reliability                                218
Bias & Fairness                            168
Privacy, Confidentiality & Infringement    223
Security & Misuse                          664
Autonomous Actions                         222
Governance, Oversight & Explainability      82
Name: count, dtype: int64


In [3]:
counts = _ec.build_counts(df)
full_months = pd.period_range(counts.index.min(), counts.index.max(), freq="M")
print(counts.shape, counts.index[0], "->", counts.index[-1])

(127, 6) 2016-01 -> 2026-07


In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
evaluate = _ec.evaluate
mlog = _ec.mlog

In [ ]:
REFERENCE_CATEGORY = _ec.REFERENCE_CATEGORY

NON_REFERENCE = [
    category for category in MECH_ORDER
    if category != REFERENCE_CATEGORY
]

FEATURE_SPECS = [
    {
        "name": "short_memory",
        "lags": [1, 2, 3],
        "windows": [],
        "rolling_std": False,
    },
    {
        "name": "medium_with_std",
        "lags": [1, 3, 6],
        "windows": [3, 6],
        "rolling_std": True,
    },
    {
        "name": "full",
        "lags": [1, 2, 3, 6, 12],
        "windows": [3, 6, 12],
        "rolling_std": True,
    },
]

ALPHA_GRID = [0.03, 0.1, 0.3, 1.0]
L1_RATIO_GRID = [0.1, 0.5, 0.9]
PSEUDOCOUNT_GRID = [0.1, 0.5, 1.0]

# Tuned after selecting the core model.
SHRINKAGE_GRID = [0.0, 0.05, 0.10, 0.20]


def smoothed_shares(count_table, pseudocount):
    smoothed = count_table.astype(float) + pseudocount
    return smoothed.div(smoothed.sum(axis=1), axis=0)


def alr_transform(shares):
    reference = shares[REFERENCE_CATEGORY].clip(lower=1e-12)

    return pd.DataFrame(
        {
            category: np.log(
                shares[category].clip(lower=1e-12) / reference
            )
            for category in NON_REFERENCE
        },
        index=shares.index,
    )


def inverse_alr(z):
    z = np.asarray(z, dtype=float)

    if z.ndim == 1:
        z = z.reshape(1, -1)

    exp_z = np.exp(np.clip(z, -30, 30))
    denominator = 1.0 + exp_z.sum(axis=1, keepdims=True)

    output = np.zeros((len(z), len(MECH_ORDER)))

    non_reference_shares = exp_z / denominator

    for j, category in enumerate(NON_REFERENCE):
        output[:, MECH_ORDER.index(category)] = non_reference_shares[:, j]

    output[:, MECH_ORDER.index(REFERENCE_CATEGORY)] = (1.0 / denominator)[:, 0]

    return output


In [ ]:
def make_feature_table(count_table, spec, pseudocount):
    shares = smoothed_shares(count_table, pseudocount)
    z = alr_transform(shares)

    features = pd.DataFrame(index=count_table.index)

    for lag in spec["lags"]:
        for category in NON_REFERENCE:
            features[f"alr_{category}_lag_{lag}"] = z[category].shift(lag)

    totals = count_table.sum(axis=1).astype(float)

    for lag in spec["lags"]:
        features[f"total_lag_{lag}"] = np.log1p(totals.shift(lag))

    for window in spec["windows"]:
        for category in NON_REFERENCE:
            shifted = z[category].shift(1)

            features[f"alr_{category}_roll_mean_{window}"] = (
                shifted.rolling(window, min_periods=window).mean()
            )

            if spec["rolling_std"]:
                features[f"alr_{category}_roll_std_{window}"] = (
                    shifted.rolling(window, min_periods=window).std()
                )

    features["month_sin"] = np.sin(
        2 * np.pi * count_table.index.month / 12
    )

    features["month_cos"] = np.cos(
        2 * np.pi * count_table.index.month / 12
    )

    features["time_index"] = np.arange(
        len(count_table),
        dtype=float
    ) / 120.0

    return features, z

In [ ]:
def fit_multitask_model(X_train, y_train, alpha, l1_ratio):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    model = MultiTaskElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        fit_intercept=True,
        max_iter=30000,
        tol=1e-6,
        random_state=RANDOM_SEED,
    )

    model.fit(X_scaled, y_train)

    return {
        "model": model,
        "scaler": scaler,
        "alpha": alpha,
        "l1_ratio": l1_ratio,
        "feature_columns": X_train.columns.tolist(),
    }


def historical_composition(count_table, pseudocount):
    totals = count_table.sum(axis=0).astype(float) + pseudocount

    return (totals / totals.sum()).reindex(MECH_ORDER).to_numpy(float)


def recursive_fold_forecast(
    train_counts,
    fold,
    spec,
    pseudocount,
    alpha,
    l1_ratio,
    shrinkage,
):
    history = train_counts.copy().astype(float)

    features, target = make_feature_table(
        history,
        spec,
        pseudocount,
    )

    usable = (
        features
        .join(target.add_prefix("target_"))
        .dropna()
    )

    feature_columns = features.columns.tolist()

    target_columns = [
        f"target_{category}"
        for category in NON_REFERENCE
    ]

    X_train = usable[feature_columns]

    y_train = usable[target_columns].copy()
    y_train.columns = NON_REFERENCE

    fitted = fit_multitask_model(
        X_train,
        y_train,
        alpha,
        l1_ratio,
    )

    prior_shares = historical_composition(
        train_counts,
        pseudocount,
    )

    recent_total = max(
        1.0,
        float(
            train_counts.sum(axis=1).tail(12).median()
        ),
    )

    output_rows = []

    yr, half = fold
    fstart, fend = _ec.fold_bounds(yr, half)

    for month in pd.period_range(fstart, fend, freq="M"):
        placeholder = pd.DataFrame(
            np.zeros((1, len(MECH_ORDER))),
            index=pd.PeriodIndex([month], freq="M"),
            columns=MECH_ORDER,
        )

        temporary_counts = pd.concat(
            [history, placeholder]
        )

        temporary_features, _ = make_feature_table(
            temporary_counts,
            spec,
            pseudocount,
        )

        current_X = temporary_features.loc[
            [month],
            feature_columns,
        ]

        if current_X.isna().any(axis=None):
            raise RuntimeError(f"Missing features for {month}")

        predicted_z = fitted["model"].predict(
            fitted["scaler"].transform(current_X)
        )

        raw_shares = inverse_alr(predicted_z)[0]

        predicted_shares = (
            (1.0 - shrinkage) * raw_shares
            + shrinkage * prior_shares
        )

        predicted_shares = np.clip(
            predicted_shares,
            1e-12,
            None,
        )

        predicted_shares /= predicted_shares.sum()

        predicted_counts = predicted_shares * recent_total

        history.loc[month, MECH_ORDER] = predicted_counts

        for category, share_hat in zip(
            MECH_ORDER,
            predicted_shares,
        ):
            output_rows.append(
                {
                    "month_period": month,
                    "category": category,
                    "predicted_share": float(share_hat),
                }
            )

    return pd.DataFrame(output_rows), fitted

In [ ]:
def attach_actuals(forecast, actual_counts):
    actual = (actual_counts.rename_axis("month_period").reset_index()
              .melt(id_vars="month_period", var_name="category", value_name="actual_count"))
    actual["actual_total"] = actual.groupby("month_period")["actual_count"].transform("sum")
    actual["actual_share"] = actual["actual_count"] / actual["actual_total"].clip(lower=1e-12)
    result = forecast.copy()
    result["month_period"] = pd.PeriodIndex(result["month_period"], freq="M")
    return result.merge(
        actual[["month_period", "category", "actual_count", "actual_share"]],
        on=["month_period", "category"], how="inner")


def score_forecast(scored):
    mae = mean_absolute_error(scored["actual_share"], scored["predicted_share"])
    rmse = mean_squared_error(scored["actual_share"], scored["predicted_share"]) ** 0.5
    log_score = (scored.groupby("month_period")
                 .apply(lambda g: mlog(g["actual_count"], g["predicted_share"]),
                        include_groups=False).mean())
    return {"mae": float(mae), "rmse": float(rmse), "log_score": float(log_score)}


def tune_core_multitask(train_counts, outer_fold):
    val_yr, val_half = _ec.prev_fold(*outer_fold)
    vstart, vend = _ec.fold_bounds(val_yr, val_half)

    inner_train = train_counts[train_counts.index < vstart].copy()
    validation_counts = train_counts[(train_counts.index >= vstart) &
                                     (train_counts.index <= vend)].copy()

    if inner_train.empty or validation_counts.empty:
        raise RuntimeError(f"No internal validation fold for {_ec.fold_label(*outer_fold)}")

    rows = []
    total_candidates = (len(FEATURE_SPECS) * len(PSEUDOCOUNT_GRID)
                        * len(ALPHA_GRID) * len(L1_RATIO_GRID))
    candidate = 0

    for spec in FEATURE_SPECS:
        for pseudocount in PSEUDOCOUNT_GRID:
            for alpha in ALPHA_GRID:
                for l1_ratio in L1_RATIO_GRID:
                    candidate += 1
                    if candidate % 12 == 0:
                        print(f"  Completed {candidate}/{total_candidates}")
                    try:
                        forecast_output, _ = recursive_fold_forecast(
                            inner_train, (val_yr, val_half),
                            spec, pseudocount, alpha, l1_ratio, shrinkage=0.0)
                        scored = attach_actuals(forecast_output, validation_counts)
                        metrics = score_forecast(scored)
                        rows.append({
                            "outer_fold": _ec.fold_label(*outer_fold),
                            "validation_fold": _ec.fold_label(val_yr, val_half),
                            "feature_set": spec["name"],
                            "lags": str(spec["lags"]),
                            "windows": str(spec["windows"]),
                            "rolling_std": spec["rolling_std"],
                            "pseudocount": pseudocount, "alpha": alpha, "l1_ratio": l1_ratio,
                            "validation_mae": metrics["mae"],
                            "validation_rmse": metrics["rmse"],
                            "validation_log_score": metrics["log_score"],
                            "status": "success",
                        })
                    except Exception as error:
                        rows.append({
                            "outer_fold": _ec.fold_label(*outer_fold),
                            "validation_fold": _ec.fold_label(val_yr, val_half),
                            "feature_set": spec["name"],
                            "lags": str(spec["lags"]),
                            "windows": str(spec["windows"]),
                            "rolling_std": spec["rolling_std"],
                            "pseudocount": pseudocount, "alpha": alpha, "l1_ratio": l1_ratio,
                            "validation_mae": np.nan, "validation_rmse": np.nan,
                            "validation_log_score": np.nan,
                            "status": f"{type(error).__name__}: {error}",
                        })
                    finally:
                        if "forecast_output" in locals(): del forecast_output
                        if "scored" in locals(): del scored
                    gc.collect()
    print()

    results = pd.DataFrame(rows)
    successful = results[results["status"] == "success"].dropna(
        subset=["validation_log_score", "validation_mae", "validation_rmse"])
    if successful.empty:
        raise RuntimeError("All core tuning candidates failed.")

    best = successful.sort_values(
        ["validation_log_score", "validation_mae", "validation_rmse", "alpha"],
        ascending=[False, True, True, False]).iloc[0]

    best_spec = next(spec for spec in FEATURE_SPECS if spec["name"] == best["feature_set"])
    best_core = {"spec": best_spec, "pseudocount": float(best["pseudocount"]),
                 "alpha": float(best["alpha"]), "l1_ratio": float(best["l1_ratio"])}
    return best_core, results

In [ ]:
def tune_shrinkage(train_counts, outer_fold, core):
    val_yr, val_half = _ec.prev_fold(*outer_fold)
    vstart, vend = _ec.fold_bounds(val_yr, val_half)

    inner_train = train_counts[train_counts.index < vstart].copy()
    validation_counts = train_counts[(train_counts.index >= vstart) &
                                     (train_counts.index <= vend)].copy()

    rows = []
    for shrinkage in SHRINKAGE_GRID:
        forecast, _ = recursive_fold_forecast(
            inner_train, (val_yr, val_half),
            core["spec"], core["pseudocount"], core["alpha"], core["l1_ratio"], shrinkage)
        scored = attach_actuals(forecast, validation_counts)
        metrics = score_forecast(scored)
        rows.append({
            "outer_fold": _ec.fold_label(*outer_fold),
            "validation_fold": _ec.fold_label(val_yr, val_half),
            "shrinkage": shrinkage,
            "validation_mae": metrics["mae"],
            "validation_rmse": metrics["rmse"],
            "validation_log_score": metrics["log_score"],
        })

    results = pd.DataFrame(rows)
    best = results.sort_values(
        ["validation_log_score", "validation_mae", "validation_rmse"],
        ascending=[False, True, True]).iloc[0]
    return float(best["shrinkage"]), results

In [ ]:
prediction_frames = []
core_tuning_frames = []
shrinkage_tuning_frames = []
fitted_models = {}
selected_models = []

for (yr, half) in _ec.TEST_FOLDS:
    fold = (yr, half); fold_str = _ec.fold_label(yr, half)
    start, end = _ec.fold_bounds(yr, half)
    print(f"\nTraining before {fold_str}; forecasting {fold_str}")

    train_counts = counts[counts.index < start].copy()
    test_counts = counts[(counts.index >= start) & (counts.index <= end)].copy()
    if train_counts.empty or test_counts.empty:
        continue

    try:
        core, core_results = tune_core_multitask(train_counts, fold)
        core_tuning_frames.append(core_results)

        best_shrinkage, shrink_results = tune_shrinkage(train_counts, fold, core)
        shrinkage_tuning_frames.append(shrink_results)

        print("Selected:", {
            "feature_set": core["spec"]["name"],
            "pseudocount": core["pseudocount"],
            "alpha": core["alpha"],
            "l1_ratio": core["l1_ratio"],
            "shrinkage": best_shrinkage,
        })

        forecast, fitted = recursive_fold_forecast(
            train_counts, fold,
            core["spec"], core["pseudocount"], core["alpha"], core["l1_ratio"], best_shrinkage)

    except Exception as error:
        print(f"{fold_str} failed: {type(error).__name__}: {error}")
        continue

    fitted_models[fold_str] = fitted
    selected_models.append({
        "fold": fold_str,
        "feature_set": core["spec"]["name"],
        "lags": str(core["spec"]["lags"]),
        "windows": str(core["spec"]["windows"]),
        "rolling_std": core["spec"]["rolling_std"],
        "pseudocount": core["pseudocount"],
        "alpha": core["alpha"],
        "l1_ratio": core["l1_ratio"],
        "shrinkage": best_shrinkage,
    })

    scored = attach_actuals(forecast, test_counts)
    scored["model"] = "multitask_elastic_net_tuned"
    scored["fold"] = fold_str
    prediction_frames.append(scored[[
        "model", "fold", "month_period", "category",
        "actual_count", "actual_share", "predicted_share"]])

if not prediction_frames:
    raise RuntimeError("No tuned Elastic Net forecasts were created.")

predictions = pd.concat(prediction_frames, ignore_index=True)
predictions["month_period"] = predictions["month_period"].astype(str)
core_tuning_results = pd.concat(core_tuning_frames, ignore_index=True)
shrinkage_tuning_results = pd.concat(shrinkage_tuning_frames, ignore_index=True)
selected_models = pd.DataFrame(selected_models)


Training before 2020-H1; forecasting 2020-H1
  Completed 12/108
  Completed 24/108
  Completed 36/108
  Completed 48/108
  Completed 60/108
  Completed 72/108
  Completed 84/108
  Completed 96/108
  Completed 108/108

Selected: {'feature_set': 'short_memory', 'pseudocount': 0.1, 'alpha': 0.03, 'l1_ratio': 0.1, 'shrinkage': 0.0}

Training before 2020-H2; forecasting 2020-H2
  Completed 12/108
  Completed 24/108
  Completed 36/108
  Completed 48/108
  Completed 60/108
  Completed 72/108
  Completed 84/108
  Completed 96/108
  Completed 108/108

Selected: {'feature_set': 'full', 'pseudocount': 1.0, 'alpha': 1.0, 'l1_ratio': 0.5, 'shrinkage': 0.0}

Training before 2021-H1; forecasting 2021-H1
  Completed 12/108
  Completed 24/108
  Completed 36/108
  Completed 48/108
  Completed 60/108
  Completed 72/108
  Completed 84/108
  Completed 96/108
  Completed 108/108

Selected: {'feature_set': 'short_memory', 'pseudocount': 0.5, 'alpha': 0.1, 'l1_ratio': 0.9, 'shrinkage': 0.2}

Training before 

In [ ]:
share_sums = predictions.groupby(["fold", "month_period"])["predicted_share"].sum()
print("Predicted monthly share-sum range:", share_sums.min(), "to", share_sums.max())

overall, by_fold = evaluate(predictions)

print("Overall tuned MultiTask Elastic Net performance")
display(overall)
print("Performance by fold")
display(by_fold)
print("Selected model by fold")
display(selected_models)

print("Feature-set selection frequency")
display(selected_models["feature_set"].value_counts()
        .rename_axis("feature_set").reset_index(name="times_selected"))
print("Selected pseudocounts")
display(selected_models["pseudocount"].value_counts().sort_index())
print("Selected shrinkage")
display(selected_models["shrinkage"].value_counts().sort_index())

Predicted monthly share-sum range: 0.9999999999999998 to 1.0000000000000002
Overall tuned MultiTask Elastic Net performance


,model,mae,rmse,mean_log_score_per_incident
0,multitask_elastic_net_tuned,0.098047,0.130576,-1.662343


Performance by fold


,model,fold,mae,rmse
0,multitask_elastic_net_tuned,2020-H1,0.188147,0.239126
1,multitask_elastic_net_tuned,2020-H2,0.117918,0.141567
2,multitask_elastic_net_tuned,2021-H1,0.109953,0.129551
3,multitask_elastic_net_tuned,2021-H2,0.114649,0.134244
4,multitask_elastic_net_tuned,2022-H1,0.091189,0.113764
5,multitask_elastic_net_tuned,2022-H2,0.104944,0.131616
6,multitask_elastic_net_tuned,2023-H1,0.101076,0.134004
7,multitask_elastic_net_tuned,2023-H2,0.111577,0.155752
8,multitask_elastic_net_tuned,2024-H1,0.080543,0.104811
9,multitask_elastic_net_tuned,2024-H2,0.069036,0.084206


Selected model by fold


...

,feature_set,times_selected
0,short_memory,6
1,full,6
2,medium_with_std,1


Selected pseudocounts


pseudocount
0.1    5
0.5    5
1.0    3
Name: count, dtype: int64

Selected shrinkage


shrinkage
0.0    9
0.1    1
0.2    3
Name: count, dtype: int64

In [ ]:
import eval_common as _ec
AXIS, OUT = _ec.AXIS, _ec.OUTPUT_DIR
MODEL = "multitask_elastic_net_tuned"

_pred = predictions.copy()
if "model" not in _pred.columns: _pred.insert(0, "model", MODEL)
_pred["month_period"] = _pred["month_period"].astype(str)
_pred.to_csv(OUT / f"predictions_{MODEL}_{AXIS}.csv", index=False)

_ov = overall.copy()
if "model" not in _ov.columns: _ov.insert(0, "model", MODEL)
_ov.to_csv(OUT / f"metrics_{MODEL}_overall_{AXIS}.csv", index=False)
by_fold.to_csv(OUT / f"metrics_{MODEL}_byfold_{AXIS}.csv", index=False)

_ss = _pred.groupby(["fold","month_period"])["predicted_share"].sum()
print("share-sum range:", round(_ss.min(),6), "to", round(_ss.max(),6))
assert {"model","fold","month_period","category","actual_count","predicted_share"} <= set(_pred.columns)
print("Saved:", f"predictions_{MODEL}_{AXIS}.csv", "+ metrics")

share-sum range: 1.0 to 1.0
Saved: predictions_multitask_elastic_net_tuned_mechanism.csv + metrics
